# 05 — Modelado

**Fase 5 del proyecto ISO 20022 en Redes DLT.**

Toma el dataset etiquetado de `data/processed/` y los indicadores de `data/external/` (FRED). Construye métricas on-chain diarias, las alinea con la serie objetivo de FRED y entrena un modelo de series temporales que **usa las métricas on-chain (con rezagos) como input para predecir el indicador de estrés financiero**. Serializa el modelo en `outputs/modelos/` y guarda un CSV de predicciones por fecha en `outputs/reportes/`.

> El test de precedencia (Granger) y el backtesting formal corresponden a la Fase 6. Aquí se produce el modelo y sus predicciones. El LSTM (`tensorflow`) es opcional y está desactivado por defecto.

# 1. Configuración

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = False  # activa guardado de modelos y CSV
USAR_LSTM: bool = False  # activa el modelo LSTM (opcional)

# 2. Imports

In [ ]:
import os
import glob
import json

import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score)

# tensorflow/keras se importan dentro de entrenar_lstm
# (opcional) para no exigirlos cuando USAR_LSTM=False

# 3. Paths y Constantes

In [ ]:
# Paths de entrada / salida
BASE_DIR: str = "tesis"
ARCHIVO_ONCHAIN: str = os.path.join(
    BASE_DIR, "data", "processed", "txs_etiquetadas.csv")
DIR_EXTERNAL: str = os.path.join(
    BASE_DIR, "data", "external")
DIR_MODELOS: str = os.path.join(
    BASE_DIR, "outputs", "modelos")
DIR_REPORTES: str = os.path.join(
    BASE_DIR, "outputs", "reportes")

# Constantes de modelado
SERIE_OBJETIVO: str = "STLFSI4"  # índice de estrés (FRED)
N_LAGS: int = 7          # rezagos de las métricas on-chain
TEST_SIZE: float = 0.2   # proporción temporal para test
MIN_FILAS: int = 30      # mínimo de filas para modelar
COLUMNAS: list[str] = [
    "tx_id", "red", "timestamp", "emisor", "receptor",
    "amount", "currency", "fee", "memo_raw",
    "memo_decoded", "iso_code", "iso_metodo",
]

# 4. Funciones

### 4.a — Carga de datos

In [ ]:
def cargar_onchain(ruta: str) -> pd.DataFrame:
    """Carga el dataset etiquetado y parsea el timestamp."""
    if not os.path.exists(ruta):
        if DEBUG:
            print(f"[DEBUG] no existe {ruta}")
        return pd.DataFrame(columns=COLUMNAS)
    df = pd.read_csv(ruta)
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], utc=True, errors="coerce")
    if DEBUG:
        print(f"[DEBUG] filas on-chain: {len(df)}")
    return df


def cargar_fred_ultimo(directorio: str) -> dict:
    """Carga el JSON de FRED más reciente del directorio."""
    archivos = sorted(glob.glob(
        os.path.join(directorio, "fred_*.json")))
    if not archivos:
        if DEBUG:
            print(f"[DEBUG] sin FRED en {directorio}")
        return {}
    with open(archivos[-1], encoding="utf-8") as f:
        return json.load(f)

### 4.b — Features on-chain y serie objetivo

In [ ]:
def construir_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega métricas on-chain diarias: nº de txs, monto y fee
    totales, y conteos por código ISO 20022.
    """
    base = df.dropna(subset=["timestamp"]).copy()
    if base.empty:
        return pd.DataFrame()
    base["fecha"] = base["timestamp"].dt.floor("D")
    agg = base.groupby("fecha").agg(
        n_txs=("tx_id", "count"),
        monto_total=("amount", "sum"),
        fee_total=("fee", "sum"),
    )
    piv = base.pivot_table(
        index="fecha", columns="iso_code",
        values="tx_id", aggfunc="count", fill_value=0)
    piv.columns = [f"n_{c}" for c in piv.columns]
    features = agg.join(piv, how="left").fillna(0)
    return features


def construir_objetivo(fred: dict, serie: str) -> pd.Series:
    """Construye la serie objetivo (nivel) desde FRED."""
    obs = fred.get(serie)
    if not obs and fred:
        obs = next(iter(fred.values()))
    if not obs:
        return pd.Series(dtype=float, name="objetivo")
    tabla = pd.DataFrame(obs)
    tabla["date"] = pd.to_datetime(tabla["date"], utc=True)
    tabla["value"] = pd.to_numeric(
        tabla["value"], errors="coerce")
    serie_obj = (
        tabla.dropna(subset=["value"])
        .set_index("date")["value"]
    )
    return serie_obj.rename("objetivo")

### 4.c — Alineación, rezagos y partición temporal

In [ ]:
def unir(features: pd.DataFrame,
         objetivo: pd.Series) -> pd.DataFrame:
    """Alinea features diarias con el objetivo (ffill)."""
    if features.empty or objetivo.empty:
        return pd.DataFrame()
    idx = features.index
    obj_diario = (
        objetivo.reindex(objetivo.index.union(idx))
        .sort_index().ffill().reindex(idx)
    )
    datos = features.copy()
    datos["objetivo"] = obj_diario
    return datos.dropna(subset=["objetivo"])


def agregar_lags(datos: pd.DataFrame, columnas: list,
                 n_lags: int) -> pd.DataFrame:
    """Agrega rezagos 1..n_lags de las columnas indicadas."""
    salida = datos.copy()
    for col in columnas:
        for k in range(1, n_lags + 1):
            salida[f"{col}_lag{k}"] = salida[col].shift(k)
    return salida.dropna()


def split_temporal(datos: pd.DataFrame,
                   test_size: float) -> tuple:
    """Partición temporal (sin mezclar) train/test."""
    corte = int(len(datos) * (1 - test_size))
    return datos.iloc[:corte], datos.iloc[corte:]

### 4.d — Entrenamiento, métricas y serialización

In [ ]:
def entrenar_rf(x_train: pd.DataFrame,
                y_train: pd.Series) -> RandomForestRegressor:
    """Entrena un Random Forest sobre las features on-chain."""
    modelo = RandomForestRegressor(
        n_estimators=200, random_state=42)
    modelo.fit(x_train, y_train)
    return modelo


def metricas_basicas(y_real, y_pred) -> dict:
    """MAE, RMSE y R2 de una predicción."""
    mse = mean_squared_error(y_real, y_pred)
    return {
        "MAE": mean_absolute_error(y_real, y_pred),
        "RMSE": mse ** 0.5,
        "R2": r2_score(y_real, y_pred),
    }


def guardar_modelo(modelo, nombre: str) -> None:
    """Serializa un modelo con joblib si procede."""
    if not SAVE_DATA:
        return
    os.makedirs(DIR_MODELOS, exist_ok=True)
    ruta = os.path.join(DIR_MODELOS, nombre)
    joblib.dump(modelo, ruta)
    if DEBUG:
        print(f"[DEBUG] modelo: {ruta}")


def entrenar_lstm(x_train: pd.DataFrame, y_train: pd.Series,
                  x_test: pd.DataFrame):
    """
    LSTM opcional (1 paso temporal). Devuelve
    (modelo, pred_train, pred_test) o None si falta la lib.
    """
    try:
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense
    except ImportError:
        if DEBUG:
            print("[DEBUG] tensorflow ausente, omito LSTM")
        return None
    n_feat = x_train.shape[1]
    xtr = x_train.values.reshape((-1, 1, n_feat))
    xte = x_test.values.reshape((-1, 1, n_feat))
    modelo = Sequential([
        LSTM(32, input_shape=(1, n_feat)),
        Dense(1),
    ])
    modelo.compile(optimizer="adam", loss="mse")
    modelo.fit(xtr, y_train.values, epochs=20, verbose=0)
    pred_tr = modelo.predict(xtr, verbose=0).ravel()
    pred_te = modelo.predict(xte, verbose=0).ravel()
    return modelo, pred_tr, pred_te

### 4.e — Armado del CSV de predicciones

In [ ]:
def armar_predicciones(x_train, y_train, x_test, y_test,
                       pred_tr, pred_te,
                       modelo: str) -> pd.DataFrame:
    """Consolida predicciones por fecha (train y test)."""
    fechas = list(x_train.index) + list(x_test.index)
    reales = list(y_train) + list(y_test)
    predichos = list(pred_tr) + list(pred_te)
    conjunto = ["train"] * len(x_train) + \
        ["test"] * len(x_test)
    return pd.DataFrame({
        "fecha": fechas,
        "y_real": reales,
        "y_pred": predichos,
        "conjunto": conjunto,
        "modelo": modelo,
    })

# 5. Proceso

In [ ]:
# 1. Cargar y construir el set supervisado
df = cargar_onchain(ARCHIVO_ONCHAIN)
fred = cargar_fred_ultimo(DIR_EXTERNAL)
features = construir_features(df)
objetivo = construir_objetivo(fred, SERIE_OBJETIVO)
unido = unir(features, objetivo)

cols_feature = [c for c in unido.columns if c != "objetivo"]
datos = agregar_lags(unido, cols_feature, N_LAGS)
if DEBUG:
    print(f"[DEBUG] filas para modelar: {len(datos)}")

In [ ]:
predicciones = pd.DataFrame()

if len(datos) < MIN_FILAS:
    if DEBUG:
        print("[DEBUG] datos insuficientes para modelar")
else:
    # 2. Partición temporal
    train, test = split_temporal(datos, TEST_SIZE)
    x_train = train.drop(columns=["objetivo"])
    y_train = train["objetivo"]
    x_test = test.drop(columns=["objetivo"])
    y_test = test["objetivo"]

    # 3. Modelo principal: Random Forest
    modelo_rf = entrenar_rf(x_train, y_train)
    pred_tr = modelo_rf.predict(x_train)
    pred_te = modelo_rf.predict(x_test)
    if DEBUG:
        print("[DEBUG] RF test:",
              metricas_basicas(y_test, pred_te))

    predicciones = armar_predicciones(
        x_train, y_train, x_test, y_test,
        pred_tr, pred_te, "random_forest")
    guardar_modelo(modelo_rf, "modelo_rf.joblib")

    # 4. Modelo opcional: LSTM
    if USAR_LSTM:
        salida = entrenar_lstm(x_train, y_train, x_test)
        if salida is not None:
            modelo_lstm, ptr, pte = salida
            pred_lstm = armar_predicciones(
                x_train, y_train, x_test, y_test,
                ptr, pte, "lstm")
            predicciones = pd.concat(
                [predicciones, pred_lstm],
                ignore_index=True)
            if SAVE_DATA:
                os.makedirs(DIR_MODELOS, exist_ok=True)
                modelo_lstm.save(os.path.join(
                    DIR_MODELOS, "modelo_lstm.keras"))

In [ ]:
# 5. Guardado del CSV de predicciones por fecha
if SAVE_DATA and not predicciones.empty:
    os.makedirs(DIR_REPORTES, exist_ok=True)
    ruta = os.path.join(DIR_REPORTES, "predicciones.csv")
    predicciones.to_csv(ruta, index=False)
    if DEBUG:
        print(f"[DEBUG] guardado: {ruta}")

if DEBUG and not predicciones.empty:
    print(predicciones.head())